In [1]:
import urllib.request
from pathlib import Path
from tqdm import tqdm
import pandas as pd

In [2]:
pdb_list = [pdb_n.strip().upper() for pdb_n in Path("./saved_tables/dimer_lst.csv").read_text().splitlines()]
print(pdb_list[:10])

['1ALL', '1B0N', '1DJ7', '1E44', '1FM0', '1FR2', '1FS0', '1H32', '1IXS', '1JQL']


# Download dimer pdbs

In [5]:
def download_pdb(pdb_id, output_dir):
    """
    Download a PDB file from RCSB PDB.
    
    Args:
        pdb_id: 4-character PDB ID (case-insensitive)
        output_dir: Path object for output directory
    
    Returns:
        True if successful, False otherwise
    """
    pdb_id = pdb_id.upper()
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    output_path = output_dir / f"{pdb_id}.pdb"
    
    # Skip if already exists
    if output_path.exists():
        return True
    
    try:
        urllib.request.urlretrieve(url, str(output_path))
        return True
    except Exception as e:
        print(f"Failed to download {pdb_id}: {e}")
        return False

In [7]:
output_dir = Path('../../PDB/dimer_db')
for pdb_id in tqdm(pdb_list, desc="Downloading PDB files"):
    download_pdb(pdb_id, output_dir)

# Create table for dimer pdbs

In [3]:
import pandas as pd
from Bio.PDB import PDBParser
import os

In [4]:
# Function to extract chain IDs from PDB (excluding water and hetero atoms)
def get_chain_ids(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb', pdb_path)
    model = next(structure.get_models())
    
    chain_ids = []
    for chain in model:
        # Check if chain has protein residues (not water or hetero)
        has_protein = any(res.id[0] == ' ' for res in chain)  # Standard residues
        if has_protein:
            chain_ids.append(chain.id)
    
    # Assuming exactly two chains for dimers
    if len(chain_ids) == 2:
        return chain_ids[0], chain_ids[1]  # First as receptor, second as ligand
    else:
        raise ValueError(f"Expected 2 chains, found {len(chain_ids)} in {pdb_path}")

In [11]:
# Initialize the DataFrame
df = pd.DataFrame(columns=['PDB', 'Receptor Chains', 'Ligand Chains', 'paths', 'affinity'])
# Populate the DataFrame
for pdb_id in tqdm(pdb_list):
    pdb_path = f'../../PDB/dimer_db/{pdb_id}.pdb'
    
    # Check if file exists
    if not os.path.exists(pdb_path):
        print(f"Warning: {pdb_path} does not exist. Skipping.")
        continue
    
    try:
        receptor_chain, ligand_chain = get_chain_ids(pdb_path)
        
        new_row = {
            'PDB': pdb_id,
            'Receptor Chains': receptor_chain,
            'Ligand Chains': ligand_chain,
            'paths': pdb_path,
            'affinity': 1
        }
        
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    
    except Exception as e:
        print(f"Error processing {pdb_id}: {e}")

# Display the DataFrame
df

 71%|███████   | 3590/5040 [01:47<00:42, 34.11it/s]

Error processing 5XSP: Expected 2 chains, found 4 in ../../PDB/dimer_db/5XSP.pdb


100%|██████████| 5040/5040 [02:32<00:00, 33.04it/s]


,PDB,Receptor Chains,Ligand Chains,paths,affinity
0,1ALL,A,B,../../PDB/dimer_db/1ALL.pdb,1
1,1B0N,A,B,../../PDB/dimer_db/1B0N.pdb,1
2,1DJ7,A,B,../../PDB/dimer_db/1DJ7.pdb,1
3,1E44,A,B,../../PDB/dimer_db/1E44.pdb,1
4,1FM0,D,E,../../PDB/dimer_db/1FM0.pdb,1
...,...,...,...,...,...
5034,7CD9,A,B,../../PDB/dimer_db/7CD9.pdb,1
5035,7CKF,A,B,../../PDB/dimer_db/7CKF.pdb,1
5036,7JR8,A,B,../../PDB/dimer_db/7JR8.pdb,1
5037,7NBI,A,B,../../PDB/dimer_db/7NBI.pdb,1


In [12]:
# add a ne row for df
new_row = {
    'PDB': '5XSP',
    'Receptor Chains': 'A',
    'Ligand Chains': 'B',
    'paths': '../../PDB/dimer_db/5XSP.pdb',
    'affinity': 1
}
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
df

,PDB,Receptor Chains,Ligand Chains,paths,affinity
0,1ALL,A,B,../../PDB/dimer_db/1ALL.pdb,1
1,1B0N,A,B,../../PDB/dimer_db/1B0N.pdb,1
2,1DJ7,A,B,../../PDB/dimer_db/1DJ7.pdb,1
3,1E44,A,B,../../PDB/dimer_db/1E44.pdb,1
4,1FM0,D,E,../../PDB/dimer_db/1FM0.pdb,1
...,...,...,...,...,...
5035,7CKF,A,B,../../PDB/dimer_db/7CKF.pdb,1
5036,7JR8,A,B,../../PDB/dimer_db/7JR8.pdb,1
5037,7NBI,A,B,../../PDB/dimer_db/7NBI.pdb,1
5038,7NDL,A,B,../../PDB/dimer_db/7NDL.pdb,1


In [14]:
df[df['Receptor Chains'].str.contains('C')]

,PDB,Receptor Chains,Ligand Chains,paths,affinity
11,1O97,C,D,../../PDB/dimer_db/1O97.pdb,1
130,1H2V,C,Z,../../PDB/dimer_db/1H2V.pdb,1
209,3DPL,C,R,../../PDB/dimer_db/3DPL.pdb,1
210,3EYQ,C,D,../../PDB/dimer_db/3EYQ.pdb,1
228,3L9J,C,T,../../PDB/dimer_db/3L9J.pdb,1
251,3YGS,C,P,../../PDB/dimer_db/3YGS.pdb,1
353,6CWY,C,D,../../PDB/dimer_db/6CWY.pdb,1
392,6UUI,C,X,../../PDB/dimer_db/6UUI.pdb,1
1853,3GWK,C,E,../../PDB/dimer_db/3GWK.pdb,1
2480,3V1T,C,D,../../PDB/dimer_db/3V1T.pdb,1


In [ ]:
#df.to_pickle('./saved_tables/dimer_table.pkl')
df.to_csv('./saved_tables/dimer_table.csv', index=False)

# Load all data together

In [2]:
wt_df = df = pd.read_csv('./saved_tables/wt_table.csv') # ~ 3k
abet_df = pd.read_csv('./saved_tables/AbEpiTope_table.csv') # ~ 0.3k
dimmer_df = pd.read_csv('./saved_tables/dimer_table.csv') # ~ 5k
swap_df = pd.read_csv('./saved_tables/swapped_table.csv') # ~ 3k
df = pd.concat([wt_df, abet_df, dimmer_df, swap_df], ignore_index=True)

In [4]:
df

,PDB,Receptor Chains,Ligand Chains,paths,affinity
0,1A22,B,A,../../PDB/SKEMPI_v2.0/1A22.pdb,12.622671
1,1A4Y,B,A,../../PDB/SKEMPI_v2.0/1A4Y.pdb,20.658646
2,1ACB,I,E,../../PDB/SKEMPI_v2.0/1ACB.pdb,14.567624
3,1AHW,C,"A, B",../../PDB/SKEMPI_v2.0/1AHW.pdb,11.547342
4,1AK4,D,A,../../PDB/SKEMPI_v2.0/1AK4.pdb,6.606704
...,...,...,...,...,...
11485,2B4J_r_5HLZ_l,A,"B,C",../../PDB/swap_pdbs/2B4J_r_5HLZ_l.pdb,0.000000
11486,2B4J_l_1G0V_r,A,"B,C",../../PDB/swap_pdbs/2B4J_l_1G0V_r.pdb,0.000000
11487,2B4J_l_5CZF_r,A,"B,C",../../PDB/swap_pdbs/2B4J_l_5CZF_r.pdb,0.000000
11488,2B4J_l_5UN7_r,A,"B,C",../../PDB/swap_pdbs/2B4J_l_5UN7_r.pdb,0.000000


In [ ]:
# check if df has duplicate PDB entries
df['PDB'].duplicated().sum(), df.isnull().sum()

(np.int64(107),
 PDB                0
 Receptor Chains    0
 Ligand Chains      0
 paths              0
 affinity           0
 dtype: int64)

In [6]:
# check if df has NAN values
df.isnull().sum()

PDB                0
Receptor Chains    0
Ligand Chains      0
paths              0
affinity           0
dtype: int64

In [7]:
# check if rows has not-found pdb files according to paths column
import os
for idx, row in df.iterrows():
    if not os.path.exists(row['paths']):
        print(f"File not found: {row['paths']}")